# EXP-24: BPTI H/D Exchange Protection Pattern
**MD-based prediction of amide protection factors**

- **Expected:** 11 protected amides (Dempsey 2001 NMR)
- **PASS:** ≥8/11 TP, ≤3 FP | **MARGINAL:** 6–7/11, ≤5 FP | **FAIL:** <6/11
- **Runs apo BPTI (4PTI) production MD on GPU**

In [ ]:
# Install OpenMM + all scientific dependencies (pip — OpenCL platform)
!pip install openmm mdtraj 'pymbar==4.0.3' mdanalysis
!pip install netCDF4 git+https://github.com/choderalab/openmmtools.git

# mpiplus stub — not on PyPI; minimal shim for imports
import os as _os, sys as _sys
_sp = _os.path.join(_sys.prefix, 'lib',
    f'python{_sys.version_info.major}.{_sys.version_info.minor}',
    'site-packages', 'mpiplus')
_os.makedirs(_sp, exist_ok=True)
with open(_os.path.join(_sp, '__init__.py'), 'w') as _f:
    _f.write(
        'class delayed_termination:\n'
        '    def __init__(self, func=None):\n'
        '        self._func = func\n'
        '    def __enter__(self): return self\n'
        '    def __exit__(self, *a): pass\n'
        '    def __call__(self, *args, **kwargs):\n'
        '        if self._func is not None: return self._func(*args, **kwargs)\n'
        '        if len(args) == 1 and callable(args[0]): return args[0]\n'
        '        return self\n'
        'def get_mpicomm(): return None\n'
        'def run_single_node(rank=0, broadcast_result=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def on_single_node(rank=0, broadcast_result=False, sync_nodes=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def distribute(func, seq, *args, send_results_to=None, sync_nodes=False, group_nodes=None):\n'
        '    results = [func(item, *args) for item in seq]\n'
        '    return list(zip(*results)) if results and isinstance(results[0], tuple) else (results, list(seq))\n'
    )
for _k in list(_sys.modules):
    if 'mpiplus' in _k:
        del _sys.modules[_k]

!pip install -q scipy matplotlib pandas requests gemmi

# Verify critical imports
import importlib
all_ok = True
for pkg in ['openmm', 'openmmtools', 'mdtraj', 'pymbar', 'MDAnalysis']:
    try:
        m = importlib.import_module(pkg)
        print(f"\u2713 {pkg} {getattr(m, '__version__', '')}")
    except ImportError as e:
        print(f"\u2717 {pkg}: {e}")
        all_ok = False

if all_ok:
    print("\n\u2705 All packages installed successfully!")
else:
    raise RuntimeError("Package installation failed \u2014 check error messages above")

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)
import sys, os, shutil, json, zipfile, logging
from pathlib import Path
import numpy as np

PIPELINE_ROOT = Path('/content/drive/MyDrive/spink7_pipeline')
assert PIPELINE_ROOT.exists()
sys.path.insert(0, str(PIPELINE_ROOT))

EXP_ID = 'EXP-24'
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/v3_gpu_results/{EXP_ID}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(f'/content/{EXP_ID}_work')
for d in ['prep','simulation','analysis','figures']:
    (WORK_DIR/d).mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO)

# ── Console log capture ──────────────────────────────────────
import io as _io
_log_buf = _io.StringIO()
_orig_stdout_write = sys.stdout.write
_orig_stderr_write = sys.stderr.write
def _stdout_log_write(data):
    _orig_stdout_write(data)
    _log_buf.write(data)
def _stderr_log_write(data):
    _orig_stderr_write(data)
    _log_buf.write(data)
sys.stdout.write = _stdout_log_write
sys.stderr.write = _stderr_log_write
# ─────────────────────────────────────────────────────────────

In [ ]:
# Checkpoint & auto-save utilities
import json, shutil, threading, time as _time
from pathlib import Path

class ExperimentCheckpoint:
    """Drive-backed phase checkpoint for resilient experiment execution."""

    def __init__(self, drive_output: Path, exp_id: str):
        self.drive_output = drive_output
        self.exp_id = exp_id
        self.path = drive_output / 'experiment_checkpoint.json'
        self.state = self._load()

    def _load(self) -> dict:
        if self.path.exists():
            with open(self.path) as f:
                return json.load(f)
        return {'exp_id': self.exp_id, 'phases': {}}

    def save(self):
        self.drive_output.mkdir(parents=True, exist_ok=True)
        with open(self.path, 'w') as f:
            json.dump(self.state, f, indent=2, default=str)

    def is_done(self, phase: str) -> bool:
        return phase in self.state['phases']

    def mark_done(self, phase: str, data: dict = None):
        self.state['phases'][phase] = {
            'timestamp': _time.strftime('%Y-%m-%d %H:%M:%S'),
            'data': data or {},
        }
        self.save()

    def get_data(self, phase: str) -> dict:
        return self.state['phases'].get(phase, {}).get('data', {})

    def summary(self):
        n = len(self.state['phases'])
        print(f'Checkpoint: {self.exp_id} — {n} phase(s) completed')
        for phase, info in self.state['phases'].items():
            print(f'  ✓ {phase} ({info["timestamp"]})')

def start_periodic_sync(work_dir: Path, drive_output: Path, interval_s: int = 300):
    """Background thread that syncs checkpoint/manifest files to Drive every N seconds."""
    stop_event = threading.Event()
    sync_patterns = ['*.chk', '*.json', '*manifest*', '*.npz', '*.npy']
    def _sync():
        while not stop_event.is_set():
            stop_event.wait(interval_s)
            if stop_event.is_set():
                break
            for pat in sync_patterns:
                for f in work_dir.rglob(pat):
                    if f.is_file() and f.stat().st_size < 500_000_000:
                        dest = drive_output / f.relative_to(work_dir)
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        try:
                            shutil.copy2(f, dest)
                        except Exception:
                            pass
    t = threading.Thread(target=_sync, daemon=True, name='drive-sync')
    t.start()
    return stop_event

def restore_from_drive(drive_output: Path, work_dir: Path):
    """On session restart, copy checkpoint/manifest files from Drive back to local."""
    restored = 0
    for pat in ['*.chk', '*manifest*', '*.json']:
        for f in drive_output.rglob(pat):
            if f.is_file():
                dest = work_dir / f.relative_to(drive_output)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(f, dest)
                    restored += 1
    if restored:
        print(f'Restored {restored} checkpoint/manifest files from Drive')

# Initialize
ckpt = ExperimentCheckpoint(DRIVE_OUTPUT, EXP_ID)
restore_from_drive(DRIVE_OUTPUT, WORK_DIR)
sync_stop = start_periodic_sync(WORK_DIR, DRIVE_OUTPUT, interval_s=300)
ckpt.summary()
print('Auto-save: checkpoint/manifest files sync to Drive every 5 min')

In [ ]:
import openmm
from openmm import unit, Platform
Platform.getPlatformByName('CUDA')
print(f'OpenMM {openmm.__version__}, CUDA ready')

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mdtraj as md
from src.config import SystemConfig, MinimizationConfig, EquilibrationConfig, ProductionConfig, KCAL_TO_KJ
from src.prep.pdb_fetch import fetch_pdb
from src.prep.pdb_clean import clean_structure
from src.prep.topology import build_topology
from src.prep.solvate import solvate_system
from src.simulate.minimizer import minimize_energy
from src.simulate.equilibrate import run_nvt, run_npt
from src.simulate.production import run_production
from src.simulate.platform import select_platform
from openmm.app import PME, ForceField, Simulation, HBonds, PDBFile
from openmm import LangevinMiddleIntegrator, XmlSerializer
print('Imports OK')

In [ ]:
# Config for apo BPTI
TEMPERATURE_K = 310.0
system_config = SystemConfig()
min_config = MinimizationConfig(max_iterations=10000)
eq_config = EquilibrationConfig(nvt_duration_ps=500.0, npt_duration_ps=1000.0, temperature_k=TEMPERATURE_K)
prod_config = ProductionConfig(duration_ns=100.0, temperature_k=TEMPERATURE_K)

# Known H/D exchange protected amides in BPTI (Dempsey 2001)
protected_residues = [4, 6, 14, 21, 22, 23, 24, 25, 49, 50, 51]
print(f'Protected amides (experimental): {protected_residues}')

In [ ]:
# Fetch and prepare apo BPTI (PDB 4PTI)
PREP_DIR = WORK_DIR / 'prep'
raw_pdb = fetch_pdb('4PTI', output_dir=PREP_DIR)
cleaned = clean_structure(raw_pdb, chains_to_keep=['A'], remove_heteroatoms=True, remove_waters=True)

topology, system, modeller = build_topology(cleaned, system_config)
modeller, n_water, _, _ = solvate_system(modeller, system_config)
print(f'Solvated: {n_water} waters')

ff = ForceField(system_config.force_field, system_config.water_model)
system = ff.createSystem(modeller.topology, nonbondedMethod=PME,
    nonbondedCutoff=1.0*unit.nanometer, constraints=HBonds, rigidWater=True)
integrator = LangevinMiddleIntegrator(TEMPERATURE_K*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
simulation = Simulation(modeller.topology, system, integrator, select_platform('CUDA'))
simulation.context.setPositions(modeller.positions)

# Save topology PDB
topo_pdb = PREP_DIR / 'apo_bpti_solvated.pdb'
with open(topo_pdb, 'w') as f:
    PDBFile.writeFile(modeller.topology, modeller.positions, f)

min_result = minimize_energy(simulation, min_config)
print(f'Minimized: {min_result["final_energy_kj_mol"]:.0f} kJ/mol')

In [ ]:
from src.simulate.production import resume_production

SIM_DIR = WORK_DIR / 'simulation'
nvt_result = run_nvt(simulation, eq_config, SIM_DIR/'nvt')
npt_result = run_npt(simulation, eq_config, SIM_DIR/'npt')
print(f'NPT: T={npt_result["avg_temperature_k"]:.1f} K')

prod_output = SIM_DIR / 'production'
prod_output.mkdir(parents=True, exist_ok=True)

if ckpt.is_done('production'):
    print('⏭ Production already completed, skipping')
    prod_result = ckpt.get_data('production')
else:
    existing_chk = sorted(prod_output.glob('*.chk'))
    if existing_chk:
        print(f'Resuming production from checkpoint: {existing_chk[-1].name}')
        prod_result = resume_production(simulation, prod_config, prod_output)
    else:
        prod_result = run_production(simulation, prod_config, prod_output)
    print(f'Production: {prod_result["total_time_ns"]:.1f} ns')
    # Save state/system to Drive
    for f_name in ['system.xml', 'npt_final_state.xml']:
        f_src = list((SIM_DIR).rglob(f_name))
        if f_src:
            shutil.copy2(f_src[0], DRIVE_OUTPUT / f_name)
    ckpt.mark_done('production', {'n_frames': prod_result['n_frames'], 'total_time_ns': prod_result['total_time_ns']})

In [ ]:
# Load trajectory for analysis
traj_files = sorted((SIM_DIR/'production').glob('*.dcd')) or sorted((SIM_DIR/'production').glob('*.xtc'))
traj = md.load(str(traj_files[0]), top=str(topo_pdb), stride=10)
protein_atoms = traj.topology.select('protein')
traj_prot = traj.atom_slice(protein_atoms)
traj_prot = traj_prot.superpose(traj_prot[0], atom_indices=traj_prot.topology.select('backbone'))
print(f'Trajectory: {traj_prot.n_frames} frames, {traj_prot.n_atoms} protein atoms')

In [ ]:
# C\u03b1 RMSF
ca_idx = traj_prot.topology.select('name CA')
rmsf_ca = md.rmsf(traj_prot, traj_prot, 0, atom_indices=ca_idx) * 10  # nm -> \u00c5
residue_ids = [traj_prot.topology.atom(idx).residue.resSeq for idx in ca_idx]
print(f'RMSF computed for {len(rmsf_ca)} residues')

In [ ]:
# Backbone NH SASA (time-averaged)
sasa_frames = md.shrake_rupley(traj_prot[::10], mode='atom')
nh_sasa_frac = {}
for res in traj_prot.topology.residues:
    if res.name == 'PRO':
        continue
    n_atoms = [a.index for a in res.atoms if a.name == 'N']
    if n_atoms:
        sasa_n = np.mean(sasa_frames[:, n_atoms[0]])
        nh_sasa_frac[res.resSeq] = sasa_n

max_sasa = max(nh_sasa_frac.values()) if nh_sasa_frac else 1.0
nh_sasa_pct = {k: v / max_sasa * 100 for k, v in nh_sasa_frac.items()}
print(f'NH SASA computed for {len(nh_sasa_pct)} residues')

In [ ]:
# Backbone H-bond occupancy
n_sample = min(traj_prot.n_frames, 500)
sample_idx = np.linspace(0, traj_prot.n_frames-1, n_sample, dtype=int)
hbond_counts = {r.resSeq: 0 for r in traj_prot.topology.residues}

for i in sample_idx:
    hbonds = md.baker_hubbard(traj_prot[i], freq=0.0)
    for d, _h, a in hbonds:
        d_name = traj_prot.topology.atom(d).name
        a_name = traj_prot.topology.atom(a).name
        if d_name == 'N' and a_name == 'O':
            d_res = traj_prot.topology.atom(d).residue.resSeq
            hbond_counts[d_res] = hbond_counts.get(d_res, 0) + 1

hbond_occ = {k: v / n_sample * 100 for k, v in hbond_counts.items()}
print(f'H-bond occupancy computed')

In [ ]:
# Protection factor proxy: SASA < 10% AND H-bond occupancy > 80%
predicted_protected = set()
for res_id in nh_sasa_pct:
    if nh_sasa_pct.get(res_id, 100) < 10 and hbond_occ.get(res_id, 0) > 80:
        predicted_protected.add(res_id)

protected_set = set(protected_residues)
tp = predicted_protected & protected_set
fn = protected_set - predicted_protected
fp = predicted_protected - protected_set
n_tp, n_fn, n_fp = len(tp), len(fn), len(fp)

if n_tp >= 8 and n_fp <= 3:
    verdict = 'PASS'
elif n_tp >= 6 and n_fp <= 5:
    verdict = 'MARGINAL'
else:
    verdict = 'FAIL'

print(f'TP: {n_tp}/11, FN: {n_fn}, FP: {n_fp}')
print(f'Predicted: {sorted(predicted_protected)}')
print(f'Verdict: {verdict}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
all_res = sorted(set(residue_ids) | set(nh_sasa_pct.keys()) | set(hbond_occ.keys()))

# RMSF
ax = axes[0]
ax.bar(residue_ids, rmsf_ca, color='steelblue', width=0.8)
for r in protected_residues:
    if r in residue_ids:
        ax.axvline(x=r, color='red', alpha=0.3, lw=1)
ax.set_ylabel('C\u03b1 RMSF (\u00c5)'); ax.set_title('RMSF Profile')

# NH SASA
ax = axes[1]
res_s = sorted(nh_sasa_pct.keys())
ax.bar(res_s, [nh_sasa_pct[r] for r in res_s], color='coral', width=0.8)
ax.axhline(y=10, color='red', ls='--', label='10% threshold')
ax.set_ylabel('NH SASA (% max)'); ax.legend()

# H-bond occupancy  
ax = axes[2]
res_h = sorted(hbond_occ.keys())
ax.bar(res_h, [hbond_occ[r] for r in res_h], color='#2ecc71', width=0.8)
ax.axhline(y=80, color='red', ls='--', label='80% threshold')
ax.set_xlabel('Residue Number'); ax.set_ylabel('H-bond Occ (%)'); ax.legend()

fig.suptitle(f'EXP-24: H/D Exchange Protection \u2014 {verdict} ({n_tp}/11 TP)', fontsize=14)
fig.tight_layout(); fig.savefig(WORK_DIR/'figures'/'results.png', dpi=150); plt.close(fig)

In [ ]:
sync_stop.set()  # Stop periodic sync before final copy

results = {
    'experiment_id': EXP_ID, 'true_positives': n_tp, 'false_negatives': n_fn,
    'false_positives': n_fp, 'predicted': sorted(list(predicted_protected)),
    'experimental': protected_residues, 'verdict': verdict,
}
with open(WORK_DIR/'results.json', 'w') as f: json.dump(results, f, indent=2)

for item in WORK_DIR.rglob('*'):
    if item.is_file():
        dest = DRIVE_OUTPUT / item.relative_to(WORK_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)

ckpt.mark_done('complete')

# ── Save full console log ────────────────────────────────────
_console_log_text = _log_buf.getvalue()
(WORK_DIR / 'console_log.txt').write_text(_console_log_text)
(DRIVE_OUTPUT / 'console_log.txt').write_text(_console_log_text)
print(f'Console log saved: {len(_console_log_text)} chars')
# ─────────────────────────────────────────────────────────────
zip_path = Path(f'/content/{EXP_ID}_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in WORK_DIR.rglob('*'):
        if item.is_file(): zf.write(item, f'{EXP_ID}/{item.relative_to(WORK_DIR)}')
files.download(str(zip_path))